# 02 — MIMIC Hourly Cohort Preparation and Leakage Checks

This notebook is the bridge from the synthetic proof-of-concept to real MIMIC data.

It expects an **hourly table** exported from MIMIC/BigQuery with one row per ICU stay per hour.

Do not silently change the SAE definition here. The `sae_onset_hour` column must come from the cohort/labeling procedure agreed for the project.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Project root
ROOT = Path.cwd().resolve().parents[1]

# Source code
sys.path.insert(0, str(ROOT / "sae_trust_aware_v2" / "src"))

from mimic_pipeline import (
    CohortConfig,
    validate_hourly_table,
    make_hourly_labels,
    assert_no_future_rows,
    add_time_features,
    patient_level_split
)

# Existing MIMIC dataset
INPUT = ROOT / "data" / "processed" / "mimic3" / "clinical_hourly_v1.csv"

print("Project root:", ROOT)
print("Input exists:", INPUT.exists())

df = pd.read_csv(INPUT)

print("Original shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
Input exists: True
Original shape: (4556, 17)
Columns:
['subject_id', 'hadm_id', 'icustay_id', 'hour', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'heart_rate', 'map', 'resp_rate', 'spo2', 'gcs_total', 'previous_gcs', 'gcs_change', 'gcs_last_observed', 'previous_observed_gcs', 'sae']


,subject_id,hadm_id,icustay_id,hour,gcs_eye,gcs_motor,gcs_verbal,heart_rate,map,resp_rate,spo2,gcs_total,previous_gcs,gcs_change,gcs_last_observed,previous_observed_gcs,sae
0,10076,198503,201006,0,NaN,NaN,NaN,100.0,NaN,32.0,93.0,NaN,NaN,NaN,NaN,NaN,0
1,10076,198503,201006,1,NaN,NaN,NaN,104.5,NaN,33.5,94.0,NaN,NaN,NaN,NaN,NaN,0
2,10076,198503,201006,2,NaN,NaN,NaN,104.5,NaN,32.5,99.5,NaN,NaN,NaN,NaN,NaN,0
3,10076,198503,201006,3,4.0,6.0,5.0,103.0,NaN,36.0,99.0,15.0,NaN,NaN,15.0,NaN,0
4,10076,198503,201006,4,NaN,NaN,NaN,90.0,NaN,32.0,99.0,NaN,15.0,NaN,15.0,15.0,0


In [4]:
required = [
    "subject_id", "stay_id", "hour",
    "sae_onset_hour",
    BASE_FEATURES = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "gcs"
]
]

validate_hourly_table(df, required)
print("Schema check: PASS")


SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3485505411.py, line 4)

In [5]:
# Adapt our existing MIMIC schema to the new pipeline

df = df.rename(columns={
    "icustay_id": "stay_id"
})

# Use the already constructed GCS measure
df["gcs"] = df["gcs_last_observed"]

# SAE onset = first hour in each ICU stay where SAE was identified
sae_onset = (
    df.loc[df["sae"] == 1]
      .groupby("stay_id")["hour"]
      .min()
      .rename("sae_onset_hour")
)

df = df.merge(
    sae_onset,
    on="stay_id",
    how="left"
)

print("Adapted shape:", df.shape)
display(
    df[
        [
            "subject_id",
            "stay_id",
            "hour",
            "gcs",
            "heart_rate",
            "map",
            "resp_rate",
            "spo2",
            "sae",
            "sae_onset_hour"
        ]
    ].head(10)
)

Adapted shape: (4556, 19)


,subject_id,stay_id,hour,gcs,heart_rate,map,resp_rate,spo2,sae,sae_onset_hour
0,10076,201006,0,NaN,100.000000,NaN,32.000000,93.000000,0,11.0
1,10076,201006,1,NaN,104.500000,NaN,33.500000,94.000000,0,11.0
2,10076,201006,2,NaN,104.500000,NaN,32.500000,99.500000,0,11.0
3,10076,201006,3,15.0,103.000000,NaN,36.000000,99.000000,0,11.0
4,10076,201006,4,15.0,90.000000,NaN,32.000000,99.000000,0,11.0
5,10076,201006,5,15.0,103.000000,NaN,35.000000,97.000000,0,11.0
6,10076,201006,6,15.0,90.000000,NaN,35.000000,99.000000,0,11.0
7,10076,201006,7,15.0,101.000000,NaN,25.000000,93.000000,0,11.0
8,10076,201006,8,15.0,107.000000,NaN,33.000000,90.500000,0,11.0
9,10076,201006,9,15.0,74.666667,NaN,22.333333,97.333333,0,11.0


In [6]:
required = [
    "subject_id",
    "stay_id",
    "hour",
    "sae_onset_hour",
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "gcs"
]

validate_hourly_table(df, required)

print("Schema check: PASS")

Schema check: PASS


In [7]:
cfg = CohortConfig(horizon=6)

df = make_hourly_labels(df, cfg)

print("6-hour future SAE distribution:")
print(df["y"].value_counts())

print("\nPositive future-SAE rows:", int(df["y"].sum()))
print("Positive rate:", round(df["y"].mean(), 4))

6-hour future SAE distribution:
y
0    4472
1      84
Name: count, dtype: int64

Positive future-SAE rows: 84
Positive rate: 0.0184


In [8]:
df = df[
    df["sae_onset_hour"].isna()
    | (df["hour"] < df["sae_onset_hour"])
].copy()

assert_no_future_rows(df, cfg)

print("Future-row leakage check: PASS")
print("Rows after filtering:", len(df))

Future-row leakage check: PASS
Rows after filtering: 1584


In [9]:
BASE_FEATURES = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "gcs"
]

df = add_time_features(
    df,
    feature_cols=BASE_FEATURES,
    group_cols=("subject_id", "stay_id"),
    history_hours=3
)

MODEL_FEATURES = []

for c in BASE_FEATURES:
    MODEL_FEATURES.extend([
        c,
        f"{c}_delta",
        f"{c}_mean3h"
    ])

print("Model features:")
print(MODEL_FEATURES)

Model features:
['heart_rate', 'heart_rate_delta', 'heart_rate_mean3h', 'map', 'map_delta', 'map_mean3h', 'resp_rate', 'resp_rate_delta', 'resp_rate_mean3h', 'spo2', 'spo2_delta', 'spo2_mean3h', 'gcs', 'gcs_delta', 'gcs_mean3h']


In [10]:
train_df, val_df, test_df = patient_level_split(
    df,
    group_cols=("subject_id", "stay_id"),
    train_frac=0.60,
    val_frac=0.20,
    seed=42
)

print("TRAIN")
print("Rows:", len(train_df))
print("Patients:", train_df["subject_id"].nunique())
print("ICU stays:", train_df["stay_id"].nunique())
print("Future SAE:", int(train_df["y"].sum()))

print("\nVALIDATION")
print("Rows:", len(val_df))
print("Patients:", val_df["subject_id"].nunique())
print("ICU stays:", val_df["stay_id"].nunique())
print("Future SAE:", int(val_df["y"].sum()))

print("\nTEST")
print("Rows:", len(test_df))
print("Patients:", test_df["subject_id"].nunique())
print("ICU stays:", test_df["stay_id"].nunique())
print("Future SAE:", int(test_df["y"].sum()))

TRAIN
Rows: 793
Patients: 14
ICU stays: 22
Future SAE: 48

VALIDATION
Rows: 338
Patients: 7
ICU stays: 7
Future SAE: 12

TEST
Rows: 453
Patients: 8
ICU stays: 9
Future SAE: 24


In [11]:
train_patients = set(train_df["subject_id"])
val_patients = set(val_df["subject_id"])
test_patients = set(test_df["subject_id"])

print("Train ∩ Validation:", len(train_patients & val_patients))
print("Train ∩ Test:", len(train_patients & test_patients))
print("Validation ∩ Test:", len(val_patients & test_patients))

assert len(train_patients & val_patients) == 0
assert len(train_patients & test_patients) == 0
assert len(val_patients & test_patients) == 0

print("\nPATIENT LEAKAGE CHECK: PASS")

Train ∩ Validation: 1
Train ∩ Test: 3
Validation ∩ Test: 1


AssertionError: 

In [12]:
OUT = ROOT / "sae_trust_aware_v2" / "data"
OUT.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(
    OUT / "train_hourly.parquet",
    index=False
)

val_df.to_parquet(
    OUT / "val_hourly.parquet",
    index=False
)

test_df.to_parquet(
    OUT / "test_hourly.parquet",
    index=False
)

print("Saved:")
print(OUT / "train_hourly.parquet")
print(OUT / "val_hourly.parquet")
print(OUT / "test_hourly.parquet")

Saved:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/train_hourly.parquet
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/val_hourly.parquet
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/test_hourly.parquet


## Expected real-data workflow

```text
MIMIC-IV
   ↓
Sepsis-3 cohort
   ↓
SAE onset definition
   ↓
hourly aggregation
   ↓
backward-looking features
   ↓
patient/stay split
   ↓
Stage 1 notebook
   ↓
Stage 2 offline RL notebook
```

The SQL/query used to construct the raw hourly table should be kept separately and version-controlled.
